# Batch Inference: Fashion Recommendations

This notebook performs batch inference using the registered models to generate
recommendations for all customers.

**Inputs:**
- catalog_name: Unity Catalog name
- schema_name: Schema containing tables and models
- model_name: Model name to use for inference (popularity_model, age_rules_model, etc.)

**Outputs:**
- Recommendations table: `{catalog}.{schema}.recommendations`

## Setup

In [0]:
import sys

# Add project root to path (go up 3 levels from notebooks/)
sys.path.append("../../../")

import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, StringType
from datetime import datetime
import mlflow

from config.widget_utils import get_widget_or_default

In [0]:
# Get parameters from bundle (passed as notebook parameters)
# Falls back to defaults when running interactively
catalog_name = get_widget_or_default("catalog_name", "jongseob_demo")
schema_name = get_widget_or_default("schema_name", "dev_fashion_recommendations")
model_name = get_widget_or_default("model_name", "popularity_model")

print(f"Catalog: {catalog_name}")
print(f"Schema: {schema_name}")
print(f"Model: {model_name}")

## Load Model

In [0]:
# Load the latest production model
model_uri = f"models:/{catalog_name}.{schema_name}.{model_name}/Production"
print(f"Loading model from: {model_uri}")

try:
    model = mlflow.pyfunc.load_model(model_uri)
    print("Model loaded successfully")
except Exception as e:
    print(f"Could not load model from Production alias, trying latest version: {e}")
    # Fallback to latest version if Production alias doesn't exist
    model_uri = f"models:/{catalog_name}.{schema_name}.{model_name}/latest"
    model = mlflow.pyfunc.load_model(model_uri)
    print(f"Loaded latest version from: {model_uri}")

## Load Customer Data

In [0]:
# Load customer data for scoring
customers_table = f"{catalog_name}.{schema_name}.customers_silver"
print(f"Loading customers from: {customers_table}")

customers_df = spark.table(customers_table)
print(f"Loaded {customers_df.count()} customers")

## Generate Recommendations

In [0]:
# TODO: Implement batch scoring logic
# This is a placeholder that should be implemented based on your model's input/output schema

print("Generating recommendations...")

# Example structure (to be implemented):
# recommendations_df = model.predict(customers_df)
# recommendations_df = recommendations_df.withColumn("created_at", F.current_timestamp())

## Save Recommendations

In [0]:
# Save recommendations to Delta table
recommendations_table = f"{catalog_name}.{schema_name}.recommendations"
print(f"Saving recommendations to: {recommendations_table}")

# TODO: Uncomment when recommendations_df is implemented
# recommendations_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable(recommendations_table)

print("Batch inference completed successfully")

## Summary

In [0]:
# Display summary statistics
print(f"""
Batch Inference Summary:
========================
Model: {model_name}
Model URI: {model_uri}
Customers Scored: {customers_df.count()}
Output Table: {recommendations_table}
Status: Completed
""")